# Matching draft-night threads to picks

The thread harvest (`03_collect_reddit.ipynb`) established *which* posts exist
in each team subreddit during each draft window. This notebook decides which
of those posts are *about a specific pick*, producing the attribution table
that comment collection and sentiment scoring build on.

Input: 192 team-year JSON files in `data/raw/reddit_threads/` and the pick
tables from `02_collect_pfr_drafts.ipynb`. Output:
`data/processed/matched_threads.csv` — one row per (pick, thread) pair, with a
confidence tier per match.

In [1]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd

RAW = Path("..") / "data" / "raw" / "reddit_threads"
PROCESSED = Path("..") / "data" / "processed"

# All six draft classes: 2021-2025 have outcomes, 2026 is the prediction set
outcomes = pd.read_csv(PROCESSED / "draft_outcomes_2021_2025.csv")
picks_2026 = pd.read_csv(PROCESSED / "draft_2026_picks.csv")
picks = pd.concat([outcomes, picks_2026], ignore_index=True)

# One row per harvested thread, tagged with the team-year it came from
records = []
for path in sorted(RAW.glob("*.json")):
    code_, year = path.stem.split("_")
    for p in json.loads(path.read_text()):
        records.append({
            "team": code_,
            "year": int(year),
            "post_id": p["id"],
            "title": p.get("title", ""),
            "num_comments": p.get("num_comments", 0),
            "created_utc": p.get("created_utc"),
        })
threads = pd.DataFrame(records)

print(f"picks: {len(picks)}   threads: {len(threads)}   "
      f"team-years: {threads.groupby(['team', 'year']).ngroups}")

picks: 1551   threads: 98231   team-years: 192


### Normalizing player names for matching

Thread titles are written by fans, not databases. Matching them to the
official `pfr_player_name` field requires normalizing both sides to a common
form, and deciding what counts as "the name" of a player.

Three problems have to be handled before any matching happens:

- **Suffixes.** "Kenneth Walker III" and "Marvin Harrison Jr." have surnames
  of Walker and Harrison. Naively taking the last token yields "iii" and "jr".
- **Accents and punctuation.** Database spellings and fan spellings diverge on
  characters like apostrophes and accents (Ja'Marr, Muñoz).
- **Case.** Titles are inconsistently capitalized.

Matching is done on the **surname**, since fans rarely write full names —
"Penning was the pick" is far more common than "Trevor Penning was the pick".
The full normalized name is kept alongside it so a full-name match can be
treated as higher-confidence than a surname-only match.

In [2]:
SUFFIXES = {"jr", "sr", "ii", "iii", "iv", "v"}

def normalize(text):
    """Lowercase, strip accents, drop punctuation, collapse whitespace."""
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def split_name(full_name):
    """Return (normalized full name, surname), suffixes removed."""
    tokens = [t for t in normalize(full_name).split() if t not in SUFFIXES]
    return " ".join(tokens), (tokens[-1] if tokens else "")

picks[["name_norm", "surname"]] = picks["pfr_player_name"].apply(
    lambda n: pd.Series(split_name(n))
)
threads["title_norm"] = threads["title"].map(normalize)

picks[["season", "team", "round", "pick", "pfr_player_name", "name_norm", "surname"]].head(10)

,season,team,round,pick,pfr_player_name,name_norm,surname
0,2021,JAX,1,1,Trevor Lawrence,trevor lawrence,lawrence
1,2021,NYJ,1,2,Zach Wilson,zach wilson,wilson
2,2021,SFO,1,3,Trey Lance,trey lance,lance
3,2021,ATL,1,4,Kyle Pitts,kyle pitts,pitts
4,2021,CIN,1,5,Ja'Marr Chase,ja marr chase,chase
5,2021,MIA,1,6,Jaylen Waddle,jaylen waddle,waddle
6,2021,DET,1,7,Penei Sewell,penei sewell,sewell
7,2021,CAR,1,8,Jaycee Horn,jaycee horn,horn
8,2021,DEN,1,9,Patrick Surtain II,patrick surtain,surtain
9,2021,PHI,1,10,DeVonta Smith,devonta smith,smith


In [3]:
# 1. Players whose suffix was stripped
suffixed = picks[picks["pfr_player_name"].str.contains(
    r"\b(?:Jr|Sr|II|III|IV|V)\.?$", regex=True, na=False)]
print(f"suffixed names: {len(suffixed)}")
print(suffixed[["pfr_player_name", "surname"]].head(8).to_string(index=False))

# 2. Two players, same team, same year, same surname
dupes = picks[picks.duplicated(["team", "season", "surname"], keep=False)]
print(f"\nsurname collisions within a team-year: {len(dupes)}")
if len(dupes):
    print(dupes.sort_values(["season", "team"])[
        ["season", "team", "round", "pick", "pfr_player_name", "surname"]
    ].to_string(index=False))

# 3. Short surnames — higher false-positive risk
short = picks[picks["surname"].str.len() <= 3]
print(f"\nsurnames of 3 characters or fewer: {short['surname'].nunique()}")
print(sorted(short["surname"].unique()))

suffixed names: 41
     pfr_player_name  surname
  Patrick Surtain II  surtain
     Greg Newsome II  newsome
   Asante Samuel Jr.   samuel
Terrace Marshall Jr. marshall
    Patrick Jones II    jones
   Michael Carter II   carter
    Earnest Brown IV    brown
    Kary Vincent Jr.  vincent

surname collisions within a team-year: 28
 season team  round  pick   pfr_player_name    surname
   2021  DEN      2    35  Javonte Williams   williams
   2021  DEN      6   219     Seth Williams   williams
   2021  LAR      4   117       Bobby Brown      brown
   2021  LAR      5   174  Earnest Brown IV      brown
   2021  NYJ      4   107    Michael Carter     carter
   2021  NYJ      5   154 Michael Carter II     carter
   2022  CHI      3    71   Velus Jones Jr.      jones
   2022  CHI      5   168     Braxton Jones      jones
   2022  GNB      1    22       Quay Walker     walker
   2022  GNB      7   249    Rasheed Walker     walker
   2022  NOR      5   161   D'Marco Jackson    jackson
   2022 

### Matching rules

The harvest window opens on April 20, days before any draft starts, so titles
can name a player *before* his pick exists — "Should we take Mendoza at 1?" is
speculation, not reaction. Three rules keep attribution honest:

1. **Timing floor.** A thread can only match a pick if it was created on or
   after the day that pick's round was held (round 1 on day one, rounds 2–3 on
   day two, rounds 4–7 on day three, all in US Eastern time). Day-level
   granularity still admits same-day speculation posted before the pick; the
   comment-collection stage tightens this further.
2. **Confidence tiers.** A title containing the player's full normalized name
   is a `full` match; a title containing only the surname is a `surname`
   match. Both are kept, tagged, so downstream stages can weigh them.
3. **Ambiguity is dropped, not guessed.** A thread whose title matches more
   than one pick at the same tier — a round-recap naming several players, or a
   bare "Williams" title when the team drafted two Williamses — is discarded.
   A full-name match beats surname matches to other picks, which is exactly
   what resolves the surname-collision pairs found above.

One thread can legitimately match one pick per team-year (official selection
thread, reaction thread, highlight thread), so a pick may accumulate several
threads. All are kept; comments are deduplicated by ID downstream.

In [4]:
# First day of each draft (all drafts run Thursday-Saturday)
DRAFT_START = {
    2021: "2021-04-29", 2022: "2022-04-28", 2023: "2023-04-27",
    2024: "2024-04-25", 2025: "2025-04-24", 2026: "2026-04-23",
}
ROUND_DAY = {1: 0, 2: 1, 3: 1, 4: 2, 5: 2, 6: 2, 7: 2}

# Epoch second at which each (year, round) becomes matchable: midnight ET
# on the day that round is held
round_open = {
    (year, rnd): (pd.Timestamp(start, tz="US/Eastern")
                  + pd.Timedelta(days=day)).timestamp()
    for year, start in DRAFT_START.items()
    for rnd, day in ROUND_DAY.items()
}

picks_by_ty = {ty: grp for ty, grp in picks.groupby(["team", "season"])}

matches, ambiguous = [], 0
for t in threads.itertuples():
    candidates = picks_by_ty.get((t.team, t.year))
    if candidates is None:
        continue

    full, surname = [], []
    for p in candidates.itertuples():
        if t.created_utc < round_open[(t.year, p.round)]:
            continue
        if re.search(rf"\b{re.escape(p.name_norm)}\b", t.title_norm):
            full.append(p)
        elif re.search(rf"\b{re.escape(p.surname)}\b", t.title_norm):
            surname.append(p)

    if len(full) == 1:
        hit, tier = full[0], "full"
    elif not full and len(surname) == 1:
        hit, tier = surname[0], "surname"
    else:
        ambiguous += bool(full or surname)
        continue

    matches.append({
        "season": t.year, "team": t.team,
        "round": hit.round, "pick": hit.pick,
        "pfr_player_name": hit.pfr_player_name,
        "post_id": t.post_id, "title": t.title,
        "created_utc": t.created_utc,
        "num_comments": t.num_comments,
        "match_tier": tier,
    })

matched = pd.DataFrame(matches)
print(f"matched thread-pick pairs: {len(matched)}")
print(f"threads dropped as ambiguous: {ambiguous}")
print(matched["match_tier"].value_counts().to_string())

matched thread-pick pairs: 16677
threads dropped as ambiguous: 581
match_tier
full       12907
surname     3770


### Coverage

Attribution only works if fanbases actually post a nameable thread per pick.
Coverage by round is the honest check: first-rounders should be near-universal,
late-round picks spottier — and any team-year at zero would point to a harvest
or naming problem rather than fan silence.

In [5]:
pick_cov = (matched.groupby(["season", "team", "round", "pick"])
            .agg(n_threads=("post_id", "count"),
                 total_comments=("num_comments", "sum"))
            .reset_index())
covered = picks.merge(
    pick_cov, on=["season", "team", "round", "pick"], how="left")
covered["has_thread"] = covered["n_threads"].notna()

print("pick coverage overall:",
      f"{covered['has_thread'].mean():.1%} of {len(covered)} picks")
print("\nby season:")
print(covered.groupby("season")["has_thread"].mean().map("{:.1%}".format).to_string())
print("\nby round:")
print(covered.groupby("round")["has_thread"].mean().map("{:.1%}".format).to_string())

zero_ty = (covered.groupby(["season", "team"])["has_thread"].sum()
           .pipe(lambda s: s[s == 0]))
print(f"\nteam-years with zero matched picks: {len(zero_ty)}")
if len(zero_ty):
    print(zero_ty.to_string())

pick coverage overall: 99.2% of 1551 picks

by season:
season
2021    98.5%
2022    99.6%
2023    99.6%
2024    99.6%
2025    99.2%
2026    98.8%

by round:
round
1    100.0%
2    100.0%
3     99.6%
4     98.6%
5     98.7%
6     99.2%
7     98.7%

team-years with zero matched picks: 0


In [6]:
# Eyeball the biggest matched threads — these should read as pick threads
cols = ["season", "team", "pfr_player_name", "title", "num_comments", "match_tier"]
matched.sort_values("num_comments", ascending=False)[cols].head(15)

,season,team,pfr_player_name,title,num_comments,match_tier
5970,2021,GNB,Amari Rodgers,Aaron Rodgers Discussion Megathread,1861,surname
625,2024,ATL,Michael Penix,[Schultz] Source: The #Falcons are drafting Mi...,1377,full
14822,2021,SFO,Trey Lance,"With the 3rd overall pick, the 49ers select: T...",1369,full
1908,2021,CAR,Jaycee Horn,"With the #8 pick in the 2021 NFL Draft, the Ca...",1329,full
6031,2021,GNB,Amari Rodgers,Aaron Rodgers Discussion Thread,1262,surname
15129,2024,SFO,Ricky Pearsall,"With the 31st overall pick, the 49ers select: ...",1227,full
11055,2021,NWE,Mac Jones,[Hughes] The #Patriots have selected Alabama Q...,1174,full
12003,2021,NYG,Kadarius Toney,Giants pick Kadarius Toney with pick #20,1077,full
5772,2021,GNB,Eric Stokes,"With the 29th Pick in the 2021 NFL Draft, the ...",1023,full
538,2023,ATL,Bijan Robinson,WITH THE 8th PICK IN THE 2023 NFL DRAFT THE AT...,1023,full


In [7]:
out_path = PROCESSED / "matched_threads.csv"
matched.sort_values(["season", "team", "pick", "created_utc"]).to_csv(
    out_path, index=False)
print(f"wrote {len(matched)} rows to {out_path}")

wrote 16677 rows to ../data/processed/matched_threads.csv
